# Customer Segmentation

See the repository README and data/README.md before running. Outputs were cleared for publication.

In [ ]:
from pathlib import Path
import os
ROOT = Path.cwd() if (Path.cwd() / "notebooks").is_dir() else Path.cwd().parent
DATA_DIR = Path(os.environ.get("PROJECT_DATA_DIR", str(ROOT / "data")))
ARTIFACT_DIR = ROOT / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

import umap
import hdbscan

from sklearn.ensemble import IsolationForest

In [ ]:
customer_features = pd.read_csv(str(DATA_DIR / "customer_features.csv"))

print(customer_features.shape)

customer_features.head()

In [ ]:
features = customer_features.drop(columns=["client_id"])

features.shape
features.head()

In [ ]:
features["spend_to_income_ratio"] = np.clip(features["spend_to_income_ratio"],features["spend_to_income_ratio"].quantile(0.01),features["spend_to_income_ratio"].quantile(0.99))

features.head()

In [ ]:
scaler = StandardScaler()

features_scaled = scaler.fit_transform(features)

pca_temp = PCA()

pca_temp.fit(features_scaled)

cum_var = np.cumsum(pca_temp.explained_variance_ratio_)

plt.figure(figsize=(8,5))

plt.plot(cum_var, marker="o")

plt.axhline(0.90, color="red")

plt.grid(True)
plt.show()


n_components_90 = np.argmax(cum_var >= 0.90) + 1
print(n_components_90)

In [ ]:
pca = PCA(n_components=min(10, *features_scaled.shape))
pca_result = pca.fit_transform(features_scaled)

print(pca_result.shape)

In [ ]:
loadings = pd.DataFrame(pca.components_.T,columns=[f"PC{i+1}" for i in range(pca.n_components_)],index=features.columns)

loadings["PC1"].abs().sort_values(ascending=False).head(10)

In [ ]:
reducer = umap.UMAP(n_components=4, n_neighbors=30, min_dist=0.1, random_state=42)
umap_result = reducer.fit_transform(pca_result)

umap_result.shape


In [ ]:
umap_result

In [ ]:
results = []

for mcs in [10,15,20,25,30,40,50]:
    for ms in [5,10,15]:
        clusterer = hdbscan.HDBSCAN(min_cluster_size=mcs,min_samples=ms)
        labels = clusterer.fit_predict(umap_result)
        mask = labels != -1

        if len(set(labels[mask])) > 1:
            score = silhouette_score(umap_result[mask],labels[mask])
            noise_pct = ((labels == -1).mean()) * 100
            results.append([mcs,ms,len(set(labels))- (1 if -1 in labels else 0),noise_pct,score])


results_df = pd.DataFrame(results,columns=["min_cluster_size","min_samples","clusters","noise_pct","silhouette"])

results_df.sort_values("silhouette",ascending=False).head(21)

# Final Model

In [ ]:
clusterer = hdbscan.HDBSCAN(min_cluster_size=30,min_samples=5)
cluster_labels = clusterer.fit_predict(umap_result)

customer_features["cluster"] = cluster_labels

customer_features["cluster"].value_counts()

In [ ]:
profile_features = customer_features.drop(columns=["client_id"])

cluster_profile = (profile_features.groupby("cluster").mean(numeric_only=True).round(2))

cluster_profile

In [ ]:
iso = IsolationForest(n_estimators=300, contamination=0.1, random_state=42)

customer_features["anomaly"] = (iso.fit_predict(features_scaled))

In [ ]:
customer_features[customer_features["anomaly"] == -1].head()

In [ ]:
plt.figure(figsize=(12,6))
sns.heatmap(
    cluster_profile,
    annot=True,
    cmap="coolwarm"
)
plt.title("Cluster Profiles")
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))

unique_clusters = sorted(set(cluster_labels))
colors = plt.cm.tab10.colors

for cluster in unique_clusters:
    mask = cluster_labels == cluster
    label = f"Noise" if cluster == -1 else f"Cluster {cluster}"
    color = "lightgrey" if cluster == -1 else colors[cluster % 10]
    plt.scatter(
        umap_result[mask, 0],
        umap_result[mask, 1],
        c=[color],
        label=label,
        alpha=0.6,
        s=20,
        edgecolors='none'
    )

plt.title("Customer Segments — UMAP Projection (colored by HDBSCAN cluster)", fontsize=14)
plt.xlabel("UMAP Dimension 1")
plt.ylabel("UMAP Dimension 2")
plt.legend(loc="best", markerscale=2, fontsize=9)
plt.tight_layout()
plt.show()

# Validation

In [ ]:
import json

In [ ]:
# Load the CSV
txn_map = pd.read_csv(str(DATA_DIR / "id_client_id.csv"))
print(txn_map.columns.tolist())
print(txn_map.head(3))

In [ ]:

with open(str(DATA_DIR / "train_fraud_labels.json")) as f:
    fraud_data = json.load(f)

target = fraud_data['target']
print(type(target))

# If it's a dict
if isinstance(target, dict):
    keys = list(target.keys())
    print("First 3 keys:", keys[:3])
    print("First 3 values:", [target[k] for k in keys[:3]])

# If it's a list
elif isinstance(target, list):
    print(target[:3])


In [ ]:
# Step 1: Load JSON and convert to DataFrame
with open(str(DATA_DIR / "train_fraud_labels.json")) as f:
    fraud_data = json.load(f)


fraud_df = pd.DataFrame(list(fraud_data['target'].items()), columns=['id', 'is_fraud'])
fraud_df['id'] = fraud_df['id'].astype(int)
fraud_df['is_fraud'] = fraud_df['is_fraud'].map({'Yes': 1, 'No': 0})

txn_map = pd.read_csv(str(DATA_DIR / "id_client_id.csv"))
txn_map = txn_map[['id', 'client_id']]

# Step 3: Merge fraud labels onto transactions
txn_fraud = txn_map.merge(fraud_df, on='id', how='inner')

# Step 4: Aggregate to customer level
customer_fraud = txn_fraud.groupby('client_id').agg(
    fraud_txn_count=('is_fraud', 'sum'),
    has_fraud=('is_fraud', 'max')   # 1 if ANY transaction was fraud
).reset_index()

print("Customers with at least 1 fraud txn:", customer_fraud['has_fraud'].sum())
print(customer_fraud.head(3))

In [ ]:
validation = customer_features[['client_id', 'anomaly']].merge(customer_fraud[['client_id', 'has_fraud']],on='client_id',how='left')

validation['has_fraud'] = validation['has_fraud'].fillna(0).astype(int)

print("=" * 45)
print("   ISOLATION FOREST — VALIDATION RESULTS")
print("=" * 45)

ct = pd.crosstab(validation['anomaly'],validation['has_fraud'],rownames=['Model Prediction'],colnames=['Actual Fraud Customer'])

ct.index = ct.index.map({1: 'Normal (1)', -1: 'Flagged Anomaly (-1)'})
print(ct)

print()
caught     = validation[(validation['anomaly'] == -1) & (validation['has_fraud'] == 1)]
missed     = validation[(validation['anomaly'] == 1)  & (validation['has_fraud'] == 1)]
false_flag = validation[(validation['anomaly'] == -1) & (validation['has_fraud'] == 0)]
total_fraud = validation['has_fraud'].sum()
total_flagged = (validation['anomaly'] == -1).sum()

print(f"Total fraud customers      : {total_fraud}")
print(f"Total flagged by model     : {total_flagged}")
print(f"Correctly caught (TP)      : {len(caught)}")
print(f"Missed fraud (FN)          : {len(missed)}")
print(f"False alarms (FP)          : {len(false_flag)}")
print(f"Detection Rate (Recall)    : {(round(len(caught)/total_fraud*100, 1) if total_fraud else 0.0)}%")
print(f"Precision                  : {(round(len(caught)/total_flagged*100, 1) if total_flagged else 0.0)}%")

In [ ]:
print(cluster_profile[['avg_transaction_amount','total_spend',
                         'transaction_count','online_txn_ratio',
                         'night_txn_ratio','weekend_txn_ratio',
                         'merchant_diversity','error_rate',
                         'spend_to_income_ratio']].to_string())